# Future work: Extract offline engine as an npm library

**Status:** Future work (discussion captured 2026-06-10)  
**Owner:** Custospark / Opiyo  
**Related:** [offline-architecture.md](./offline-architecture.md), [offline/README.md](../src/renderer/app/store/offline/README.md)

This notebook documents a proposed path to turn Custosell's offline-first layer into a reusable npm package (e.g. `@opiyo/offline-core`) that other projects can install. **No implementation has started** — this is planning material for when a second consumer project justifies the extraction cost.

## 1. Motivation

Custosell's renderer implements a mature offline stack:

- Write-local, read-merged mutations
- IndexedDB mutation queue with tiered sync on reconnect
- Durable catalog snapshots that survive logout
- Device login with silent session upgrade (no 401 race on reconnect)
- Connectivity model: `offline` | `slow` | `online`

The `src/renderer/app/store/offline/` tree was recently reorganized into module-aligned subfolders (`core`, `auth`, `sales`, `inventory`, `sync`, etc.) — a good foundation for extraction.

**Goal:** Publish a **framework + plugins** library, not a POS-in-a-box. Custosell remains the reference implementation and first consumer.

## 2. Feasibility verdict

| Question | Answer |
|----------|--------|
| Is it possible? | **Yes** |
| Can we copy `offline/` as-is to npm? | **No** — ~50% is Custosell domain logic |
| Recommended approach? | Extract a **core kernel**; keep POS entities as **plugins** in the app |
| When to start? | When a **second real project** needs the same patterns, or Custosell maintenance cost of duplication grows |

The hardest coupling is not IndexedDB — it is **business rules** baked into `syncEngine.ts` (shift-close gating, refund blocking, negative ID remap, Laravel API URL shapes).

## 3. Current codebase split

Rough breakdown of `src/renderer/app/store/offline/` (~60 files):

| Layer | ~Share | Reusability | Key files |
|-------|--------|-------------|----------|
| **Generic kernel** | 25–30% | High | `sync/mutationQueue.ts`, `core/offlineDb.ts`, `auth/secureStorage.ts`, `catalogs/serverCatalogStore.ts`, `core/offlineReadStrategy.ts` |
| **Orchestration** | 15–20% | Medium (needs API redesign) | `sync/syncCoordinator.ts`, `sync/syncProgressReporter.ts`, `sync/syncConstants.ts` |
| **Domain + API rules** | 50–55% | Low | `sync/syncEngine.ts`, all `completeOffline*` / `local*Store`, `sales/receiptGenerator.ts`, `inventory/stockLedger.ts` |

### Coupling hotspots

- `syncEngine.ts` imports Custosell types from `modules/expenses`, `modules/settings`, Redux `authSlice`, and `axiosConfig`
- `syncCacheRefresh.ts` / `auth/sessionRefresh.ts` know React Query key shapes
- `offlineDb.ts` hardcodes 17+ object stores (`localSales`, `localShifts`, …) under `CustosellOffline` v12
- Connectivity reads Redux `networkSlice.systemStatus` via `offlineQueryUtils.ts`

## 4. Proposed package architecture

Layered npm packages with **zero React/Redux in core**:

```text
@opiyo/offline-core          ← mutation queue, IDB registry, snapshots, coordinator shell
@opiyo/offline-react         ← optional peer: React Query gating, useOfflineSync hooks
@opiyo/offline-auth          ← optional: device login, silent session upgrade

custosell-offline-entities   ← stays in Custosell (or private package): sales, shifts, products plugins
```

### Core responsibilities (`@opiyo/offline-core`)

1. **Mutation queue** — enqueue POST/PUT/PATCH/DELETE, retry, stale `syncing` recovery
2. **IndexedDB registry** — app defines stores/schemas; library does not ship `localSales`
3. **Catalog snapshots** — generic `{namespace}:{tenant}:{kind}` blob persistence
4. **Connectivity** — `offline | slow | online` via injectable probe (not Redux-only)
5. **Read strategy** — server-first with client fallback on network failure
6. **Sync coordinator** — runs registered entity handlers in dependency order
7. **Secure secrets** — AES-GCM wrapper for tokens/passwords (Web Crypto)
8. **ID remap** — map local negative IDs → server IDs after sync

### What stays in Custosell forever

- Entity-specific `local*Store` implementations
- URL matchers and sync handlers for `/sales`, `/shifts`, `/products`, …
- Receipt generation, stock ledger overlay rules
- React Query invalidation key lists
- POS-specific auth slice integration

## 5. Entity plugin pattern (target API sketch)

Consumer apps register entities instead of forking `syncEngine.ts`:

```typescript
import { createOfflineEngine, registerEntity } from '@opiyo/offline-core';

const engine = createOfflineEngine({
  dbName: 'MyAppOffline',
  transport: axiosAdapter,           // or fetch
  getNetworkStatus: () => status,    // injected, not Redux
  onSyncProgress: (p) => dispatch(p),
});

registerEntity(engine, {
  name: 'sales',
  tier: 3,
  dependsOn: ['shifts', 'products'],
  match: (m) => m.method === 'POST' && m.url === '/sales',
  sync: syncSaleMutation,
  remapIds: (map) => remapSaleRefs(map),
});

// Reconnect
await engine.upgradeSessionIfNeeded();  // optional auth package
await engine.syncPendingIfOnline();
```

Custosell's current `sync/syncEngine.ts` (~775 lines) becomes a **bundle of entity plugins** colocated with domain modules.

## 6. Package naming options

| Name | Notes |
|------|-------|
| `@opiyo/offline-core` | **Recommended** — clear, extensible, scoped |
| `@opiyo/offline-kit` | Signals toolkit, not finished product |
| `@opiyo/write-local` | Memorable; matches "write-local, read-merged" principle |
| `opiyo-offline-sync` | Fine unscoped; slightly narrow (reads, auth, snapshots are also in scope) |
| `@custosell/offline-engine` | Brand-tied; weaker signal for external reuse |

**Suggestion:** publish `@opiyo/offline-core` first; add `@opiyo/offline-react` and `@opiyo/offline-auth` only when a second app needs them.

## 7. Phased implementation plan

| Phase | Deliverable | Effort (rough) |
|-------|-------------|----------------|
| **0. Design API** | Plugin registry, transport interface, schema config contract | 1 week |
| **1. Extract core** | Queue, IDB helpers, snapshots, secure storage, connectivity interface | 2–3 weeks |
| **2. Refactor Custosell** | Replace direct imports; entity plugins in-app; prove API | 2–3 weeks |
| **3. React adapter** | `useOfflineSync`, query online gating as peer dep | 1 week |
| **4. Auth module** | Optional device login + silent upgrade package | 1–2 weeks |
| **5. Publish** | Monorepo, `tsup`, tests, README, semver `0.x` | 1 week |

**Total:** ~6–8 weeks for a trustworthy `0.1.0` usable in a second project.

### Suggested repo layout

```text
opiyo-offline/                 # separate repo or monorepo root
  packages/
    core/                      # @opiyo/offline-core
    react/                     # @opiyo/offline-react (optional)
    auth/                      # @opiyo/offline-auth (optional)
  examples/
    minimal-spa/               # tiny demo consumer
  package.json                 # pnpm workspaces
```

### Peer dependencies (core)

- `idb` (required)
- `axios` or native `fetch` via injectable transport
- `@tanstack/react-query` (react package only)
- No Electron, no Redux in core

## 8. Risks and mitigations

| Risk | Mitigation |
|------|------------|
| **IDB schema migrations** differ per app | Library provides migration helpers; apps own `dbName`, version, store defs |
| **Auth models differ** across products | Keep `@opiyo/offline-auth` optional; core works with bearer token injection |
| **Sync ordering** is domain-specific | Library provides tiers + `dependsOn`; apps register handlers |
| **Early API churn** on `0.x` | Do not extract until second project validates; semver strictly |
| **Over-abstraction** | Start with smallest core (queue + coordinator shell); expand on demand |
| **Custosell regression** during refactor | Extract behind interfaces first; swap imports incrementally; `tsc -b` + vera gates |
| **Testing gap** | Unit tests in core; Custosell manual offline matrix as integration proof |

## 9. Decision matrix

| Approach | Verdict |
|----------|---------|
| Copy entire `offline/` folder → npm | ❌ Too coupled; painful for other projects |
| Extract core + plugins; Custosell as consumer | ✅ Best long-term |
| Wait until second project needs it | ⚠️ Valid if no near-term reuse |
| Publish POS-specific sync rules in the library | ❌ Wrong abstraction level |

### Pragmatic recommendation

1. **Now:** Keep improving Custosell offline in-app; maintain `offline/README.md` and architecture docs.
2. **Trigger:** Second app (or client project) needs the same queue + reconnect + snapshot patterns.
3. **First publish:** `@opiyo/offline-core` only — mutation queue, IDB registry, coordinator shell, read strategy.
4. **Proof:** Custosell consumes the published package with entity plugins still in-repo.

## 10. Migration map (current → future packages)

| Current path (Custosell) | Future home |
|--------------------------|-------------|
| `offline/sync/mutationQueue.ts` | `@opiyo/offline-core` |
| `offline/core/offlineDb.ts` | `@opiyo/offline-core` (generic schema registry) |
| `offline/catalogs/serverCatalogStore.ts` | `@opiyo/offline-core` |
| `offline/auth/secureStorage.ts` | `@opiyo/offline-core` or `@opiyo/offline-auth` |
| `offline/core/offlineReadStrategy.ts` | `@opiyo/offline-core` |
| `offline/sync/syncCoordinator.ts` | `@opiyo/offline-core` (generic coordinator) |
| `offline/sync/syncEngine.ts` | **Custosell plugins** (split by entity) |
| `offline/sales/*`, `inventory/*`, … | **Custosell plugins** |
| `offline/auth/sessionUpgrade.ts` | `@opiyo/offline-auth` (optional) |
| `app/store/hooks/useOfflineSync.ts` | `@opiyo/offline-react` |
| `app/store/hooks/useSyncQueryOnlineStatus.ts` | `@opiyo/offline-react` |

## 11. Open questions (to resolve before Phase 0)

1. **Monorepo vs separate repo** — same GitHub org as Custosell, or standalone `opiyo-offline`?
2. **Minimum v0 API** — which three exports are non-negotiable on day one?
3. **Transport** — standardize on `fetch` only, or ship axios adapter?
4. **License** — MIT for npm adoption?
5. **Second consumer** — which project validates the API first (internal tool, client app, new product)?
6. **Electron** — confirm library stays renderer-only; no main-process coupling.

---

*Documented from architecture discussion following the offline folder modularization commit (`d32a409`). Update this notebook when Phase 0 design begins.*